# 00. 코드 읽기 가이드 — 처음 오신 팀원용

이 노트북은 저장소가 어떻게 생겼고, 어떤 순서로 돌아가며, 어디를 고치면 무엇이 바뀌는지를 설명합니다.
셀을 위에서부터 하나씩 실행(Shift+Enter)하면서 읽으세요. 10분이면 끝납니다.

## 큰 그림: 세 단계

| 단계 | 명령 | 입력 → 출력 | 코드 |
|---|---|---|---|
| ① 결합 | `make data` | `data/raw/` 번들 8개 → `data/processed/bloom.parquet` | `src/data_ai/bloom.py` |
| ② 학습 | `make train` | parquet → 모델, `reports/submission.csv`, `reports/metrics.csv` | `scripts/train.py`, `src/data_ai/features.py`, `models.py` |
| ③ 확인 | `make evaluate`, `make score` | 지표 표, 제출 형식 검사 | `scripts/evaluate.py`, `scripts/score_submission.py` |

설정은 전부 `configs/bloom.yaml` 한 파일에 있습니다. 열 이름 매핑, 집계 창(7·14·28일), 지점별 상류 댐 목록이 여기 있습니다.


## 1. 결합 결과 열어 보기
(지점, 측정일) 한 행입니다. 열이 126개라 많아 보이지만 접두사로 묶으면 다섯 덩어리뿐입니다.

In [1]:
import pandas as pd
from data_ai.config import PROCESSED_DIR
pd.set_option("display.max_columns", 40, "display.width", 160)

df = pd.read_parquet(PROCESSED_DIR / "bloom.parquet")
print(df.shape)
groups = {
    "조류 원본·lag": [c for c in df.columns if c.startswith(("cyano", "cells_", "water_temp", "chl_a")) or c in ["ph","do","transparency","turbidity","mc_lr","dominant_genus","days_since_prev"]],
    "기상 집계 (w_)": [c for c in df.columns if c.startswith("w_")],
    "댐 집계 (d_)": [c for c in df.columns if c.startswith("d_")],
    "정답·분할": ["target_date","lag_days","split","target_cells","target_valid","is_test","log10_target_cells"],
    "달력·파생": ["year","month","doy","current_level","target_level","log10_cells"],
}
for k, v in groups.items():
    print(f"{k:14s} {len(v):3d}개  예) {v[:6]}")

(2939, 126)
조류 원본·lag       27개  예) ['cyano_cells', 'cells_microcystis', 'cells_anabaena', 'cells_oscillatoria', 'cells_aphanizomenon', 'water_temp']
기상 집계 (w_)      46개  예) ['w_temp_mean', 'w_temp_mean_mean7d', 'w_temp_mean_mean14d', 'w_temp_mean_mean28d', 'w_temp_max', 'w_temp_max_max7d']
댐 집계 (d_)       34개  예) ['d_inflow', 'd_inflow_mean7d', 'd_inflow_sum7d', 'd_inflow_mean14d', 'd_inflow_sum14d', 'd_inflow_mean28d']
정답·분할            7개  예) ['target_date', 'lag_days', 'split', 'target_cells', 'target_valid', 'is_test']
달력·파생            6개  예) ['year', 'month', 'doy', 'current_level', 'target_level', 'log10_cells']


## 2. 한 행을 끝까지 읽기
아래는 물금·매리 2025년 8월의 한 행입니다. 이 행에서 모델이 보는 것(입력)과 맞혀야 하는 것(정답)을 구분해 보세요.
- `cyano_cells`: 기준일 세포수. `cyano_cells_lag1`: 직전 측정값.
- `w_temp_mean_mean28d`: 기준일 포함 과거 28일 평균 기온. `w_precip_sum7d`: 과거 7일 강수 합.
- `d_outflow_mean7d`: 상류 댐들의 7일 평균 방류량 합. `d_level_anom`: 대표 댐 저수위의 월별 평년 대비 편차.
- `target_cells`: 정답(+7일 최근접 측정). `split`: train/valid/test.

In [2]:
row = df[(df.station_id == "MGMR") & (df.date.astype(str).str.startswith("2025-08"))].iloc[0]
show = ["station_id","date","split","cyano_cells","cyano_cells_lag1","water_temp","chl_a",
        "w_temp_mean_mean28d","w_precip_sum7d","d_outflow_mean7d","d_level_anom","target_date","target_cells"]
row[show]

station_id                            MGMR
date                   2025-08-04 00:00:00
split                                 none
cyano_cells                        24439.0
cyano_cells_lag1                     990.0
water_temp                            30.1
chl_a                                 33.7
w_temp_mean_mean28d              27.903571
w_precip_sum7d                        20.7
d_outflow_mean7d                   220.389
d_level_anom                          2.68
target_date                            NaT
target_cells                           NaN
Name: 2866, dtype: object

## 3. "기준일 이후 정보를 쓰지 않는다"는 어떻게 보장되나
두 군데에서 막습니다.
1. **결합 단계** (`bloom.py`): 일 자료 집계는 `rolling`(과거 창)이고, 결합은 `merge_asof(direction="backward")`로 기준일 이전 날짜만 붙습니다.
2. **입력 선택** (`features.py`): 입력으로 쓸 수 있는 열을 **허용 목록**으로 정해 두어, `target_`이 들어간 열이나 날짜 열은 애초에 못 들어갑니다. 섞이면 실행이 멈춥니다.

아래 셀은 실제 입력 열 목록을 뽑아 정답 관련 열이 없는지 확인합니다.

In [3]:
from data_ai.features import feature_columns
cols = feature_columns(df)
print(len(cols), "개 입력 열")
print("정답/분할/날짜 열 포함 여부:", any(("target" in c) or c in ("split","date","target_date") for c in cols))
print(cols[:15], "...")

112 개 입력 열
정답/분할/날짜 열 포함 여부: False
['station_id', 'cyano_cells', 'cells_microcystis', 'cells_anabaena', 'cells_oscillatoria', 'cells_aphanizomenon', 'water_temp', 'ph', 'do', 'transparency', 'turbidity', 'chl_a', 'mc_lr', 'dominant_genus', 'days_since_prev'] ...


## 4. 결측 대체 전략 3안
`apply_missing_strategy(df, 전략, train_mask)`. 중앙값 같은 통계는 **train_mask 행에서만** 계산합니다(검증·평가 정보 차단).
아래는 세 전략을 적용했을 때 수온 결측이 어떻게 바뀌는지 보여 줍니다.

In [4]:
from data_ai.features import apply_missing_strategy, STRATEGIES
train_mask = df["target_valid"] & df["split"].eq("train")
for s in STRATEGIES:
    out = apply_missing_strategy(df, s, train_mask)
    print(f"{s:18s} 수온 결측 {out['water_temp'].isna().mean():.1%}", "| 결측 표시 열:", [c for c in out.columns if c.endswith('_isna')][:3])

native             수온 결측 13.7% | 결측 표시 열: []
ffill_flag         수온 결측 0.0% | 결측 표시 열: ['cyano_cells_isna', 'log10_cells_isna', 'water_temp_isna']
site_month_median  수온 결측 0.0% | 결측 표시 열: ['cyano_cells_isna', 'log10_cells_isna', 'water_temp_isna']


## 5. 지표 함수 직접 써 보기
공식 지표는 `bloom_metrics(실제, 예측)` 한 함수로 다 나옵니다. "다음 주도 이번 주와 같다"(persistence)를 검증 2025에 적용해 봅니다.

In [5]:
from data_ai.evaluate import bloom_metrics
v = df[(df.split == "valid") & df.target_valid]
persist = v["cyano_cells"].fillna(v["cyano_cells_lag1"])
pd.Series(bloom_metrics(v["target_cells"], persist)).round(3)

n                    253.000
rmse_log10             0.883
mae_log10              0.583
r2                    -0.282
nse                   -0.282
f1_over_threshold      0.727
dtype: float64

## 6. 학습된 모델 불러와 한 행 예측하기
`make train`을 먼저 실행해야 `models/lgb_final.joblib`이 생깁니다. 모델은 log10(세포수+1)을 예측하므로 원 단위로 되돌립니다.

In [6]:
import json
from data_ai.models import load_model, predict_cells
from data_ai.features import to_matrix
from data_ai.config import MODELS_DIR
meta = json.loads((MODELS_DIR / "train_meta.json").read_text(encoding="utf-8"))
model = load_model("lgb_final")
d = apply_missing_strategy(df, meta["strategy"], df["split"].isin(["train","valid"]) & df["target_valid"])
test = d[d["is_test"]]
x, _ = to_matrix(test, meta["features"])
pred = predict_cells(model, x)
out = test[["station_id","date","cyano_cells"]].assign(pred_cells=pred.round().astype(int))
print("전략:", meta["strategy"], "| 반복:", meta["best_iter"])
out.head(8)

전략: ffill_flag | 반복: 131


,station_id,date,cyano_cells,pred_cells
596,CS,2026-05-04,941.0,548
597,CS,2026-05-11,467.0,248
598,CS,2026-05-18,1821.0,1781
599,CS,2026-05-26,711.0,889
600,CS,2026-06-01,4877.0,3806
601,CS,2026-06-08,7280.0,5731
602,CS,2026-06-15,18956.0,11534
603,CS,2026-06-22,18836.0,10374


## 7. 자주 고치는 곳

| 하고 싶은 것 | 고칠 파일 | 위치 |
|---|---|---|
| 집계 창을 7·14·28일에서 바꾸기 | `configs/bloom.yaml` | `weather.windows`, `dam.windows` |
| 지점별 상류 댐 바꾸기 | `configs/bloom.yaml` | `dam.station_dams`, `dam.primary_dam` |
| 새 피처 추가 | `src/data_ai/features.py` | `add_dynamics_features()` 참고. 허용 목록(`ALLOW_*`)에 이름 규칙 맞추기 |
| 모델 파라미터 | `src/data_ai/models.py` | `DEFAULT_PARAMS` |
| 새 실험 추가 | `scripts/experiments.py` | `run_variant()`에 분기 추가 |

고친 뒤에는 항상 `python -m pytest -q`(테스트)와 `make train`(재학습)을 돌려 숫자가 어떻게 바뀌는지 `reports/metrics.csv`로 확인하세요.

## 8. 용어 한 줄 정리
- **lag**: 이전 측정값. lag1 = 직전, lag2 = 그 전.
- **rolling 집계**: 기준일 포함 과거 N일의 평균·합. 미래는 절대 포함하지 않음.
- **persistence / climatology**: "지난번 값 그대로" / "그 지점 그 달의 평소 값". 모델이 넘어야 할 바닥선.
- **rmse_log10**: log10(세포수+1)로 바꾼 뒤의 RMSE. 공식 지표.
- **R² / NSE**: 원 단위에서 "평균값 예측보다 얼마나 나은가". 봉우리 몇 개가 좌우함.
- **early stopping**: 검증 오차가 더 안 줄면 학습을 멈추는 것.
- **누수(leakage)**: 예측 시점에 알 수 없는 정보가 입력에 섞이는 것. 대회에서는 실격 사유.
